# SpottingSalmon
YOLO model logging script
- To register the final model as an MLFlow model


In [0]:
%pip install --quiet numpy==1.26.4  mlflow ultralytics

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import ultralytics
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, TensorSpec, DataType, ColSpec
import numpy as np
from ultralytics import YOLO
from pyspark.sql import functions as F

### 1. Model logging

In [0]:
# Run ID and artifact relative path
run_id = "cc1ca4e121884869ac75efcba1a0af4a"
artifact_path = "weights/best.pt"

#model_path = f"runs:/{run_id}/{artifact_path}"
# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

In [0]:
import os

video_path = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
print("Video exists?", os.path.exists(video_path))

In [0]:
import cv2

cap = cv2.VideoCapture(video_path)
print("Opened?", cap.isOpened())

success, frame = cap.read()
print("First frame read:", success)
cap.release()

In [0]:
import os

video_path = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"  # <-- update if needed
print("Video exists?", os.path.exists(video_path))

In [0]:
import mlflow.pyfunc
import pandas as pd
from ultralytics import YOLO
import cv2
import os
import shutil
import uuid
from databricks.sdk import WorkspaceClient


def _log(msg: str):
    # Single logger (stdout) so logs don't appear twice in Databricks
    print(msg, flush=True)


def download_uc_file_to_tmp(w: WorkspaceClient, uc_path: str) -> str:
    """
    Downloads a UC Volume file (e.g. /Volumes/...) to a local temp path under /tmp.
    This is needed in Model Serving because the serving container may not have direct
    filesystem access to /Volumes.
    Supports both newer and older databricks-sdk versions.
    """
    # Use a unique temp path per request to avoid overwriting in concurrent serving
    local_path = f"/tmp/{uuid.uuid4().hex}.mp4"
    _log(f"[INFO] Downloading UC file: {uc_path}")

    files_api = w.files

    # Newer SDKs provide download_to(...)
    if hasattr(files_api, "download_to"):
        files_api.download_to(uc_path, local_path)
    else:
        # Older SDKs provide download(...) returning an object with .contents (binary stream)
        resp = files_api.download(uc_path)
        stream = getattr(resp, "contents", None)
        if stream is None:
            raise RuntimeError("Files API response missing `.contents` attribute (SDK too old).")

        with stream as f, open(local_path, "wb") as out:
            shutil.copyfileobj(f, out, length=1024 * 1024)  # copy in ~1MB chunks

    # Log a quick confirmation that download really happened
    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    _log(f"[INFO] Download complete -> {local_path} ({size_mb:.2f} MB)")
    return local_path


def read_video_frames(video_path: str, w: WorkspaceClient | None = None):
    """
    Generator that yields (frame_number, frame) from the video.
    - Notebook mode: reads directly from the given path.
    - Serving mode: downloads the UC Volume file to /tmp first, then reads it.
    """
    _log(f"[INFO] Reading video: {video_path}")

    cleanup = False
    local_path = video_path

    # In serving, use Files API to make the video available locally
    if w is not None:
        local_path = download_uc_file_to_tmp(w, video_path)
        cleanup = True
    else:
        # In notebooks, ensure the path is actually accessible
        if not os.path.exists(local_path):
            raise FileNotFoundError(f"Video path not accessible: {local_path}")

    cap = cv2.VideoCapture(local_path)

    # Fail fast if OpenCV can't open the file (bad path, corrupted file, missing codecs, etc.)
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {local_path}")

    # Log basic metadata to confirm the file opened correctly
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    _log(f"[INFO] Video opened OK -> {local_path} (frames={frame_count}, fps={fps:.2f})")

    frame_number = 0
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                _log(f"[INFO] End of video at frame {frame_number}")
                break

            yield frame_number, frame
            frame_number += 1
    finally:
        # Always release the capture and remove temp file in serving mode
        cap.release()
        if cleanup:
            try:
                os.remove(local_path)
                _log(f"[INFO] Temp file removed -> {local_path}")
            except Exception as e:
                _log(f"[WARN] Could not remove temp file {local_path}: {e}")


def convert_yolo_to_dets(result):
    """
    Converts a Ultralytics YOLO Results object into a list of plain Python dicts
    (one per bounding box), which is easy to return as a pandas DataFrame.
    """
    detections = []

    # No detections for this frame
    if result is None or result.boxes is None or len(result.boxes) == 0:
        return detections

    boxes = result.boxes
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()

    for (x1, y1, x2, y2), conf in zip(xyxy, confs):
        detections.append({
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "confidence": float(conf)
        })

    return detections


class FishVideoDetector(mlflow.pyfunc.PythonModel):
    """
    MLflow PyFunc wrapper:
    - Expects input DataFrame with a column 'fish' containing video paths.
    - Returns a DataFrame of detections: one row per detected object per frame.
    """

    def load_context(self, context):
        # Load the YOLO checkpoint artifact that was logged with the MLflow model
        _log("[INFO] Loading YOLO model")
        self.model = YOLO(context.artifacts["checkpoint"])

        # In serving, DATABRICKS_HOST and DATABRICKS_TOKEN are available and allow Files API access
        host = os.environ.get("DATABRICKS_HOST")
        token = os.environ.get("DATABRICKS_TOKEN")

        if host and token:
            self.w = WorkspaceClient(host=host, token=token)
            _log("[INFO] Running in Model Serving mode")
        else:
            self.w = None
            _log("[INFO] Running in Notebook mode")

    def predict(self, context, model_input: pd.DataFrame):
        """
        Runs detection for each video path in model_input['fish'].
        Logs key steps so serving logs show whether download/open/inference succeeded.
        """
        _log(f"[INFO] Starting prediction. Columns: {list(model_input.columns)} Rows: {len(model_input)}")

        # Input validation: must have 'fish' column
        if "fish" not in model_input.columns:
            _log("[ERROR] Missing 'fish' column in input.")
            return pd.DataFrame([])

        all_rows = []

        for idx, video_path in enumerate(model_input["fish"]):
            _log(f"[INFO] Processing row {idx}: {video_path}")

            try:
                video_name = os.path.basename(video_path)
                detections_for_video = 0

                for frame_number, frame in read_video_frames(video_path, w=self.w):
                    # Run YOLO; try to suppress Ultralytics verbose if supported
                    try:
                        result = self.model(frame, verbose=False)
                    except TypeError:
                        result = self.model(frame)

                    # Ultralytics may return a list; normalize to a single Results object
                    if isinstance(result, list):
                        result = result[0] if len(result) > 0 else None

                    dets = convert_yolo_to_dets(result)

                    for det in dets:
                        all_rows.append({
                            "fish_id": None,
                            "video": video_name,
                            "frame": frame_number,
                            **det
                        })
                        detections_for_video += 1

                _log(f"[INFO] Finished video: {video_name}. Detections: {detections_for_video}")

            except Exception as e:
                # Keep processing other rows even if one video fails
                _log(f"[ERROR] Failed processing {video_path}: {e}")

        _log(f"[INFO] Prediction complete. Total detections: {len(all_rows)}")
        return pd.DataFrame(all_rows)








In [0]:
from databricks.sdk import WorkspaceClient
import os

# Read env vars
host = os.environ.get("DATABRICKS_HOST")
token = os.environ.get("DATABRICKS_TOKEN")

print("HOST:", host)
print("TOKEN:", token)
print("TOKEN length:", len(token) if token else None)

# Try connecting to Databricks API using WorkspaceClient
try:
    w = WorkspaceClient(host=host, token=token)
    me = w.current_user.me()
    print("Connection successful!")
    print("Authenticated as:", me.user_name)
except Exception as e:
    print("Connection failed:", e)

In [0]:
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

input_example = pd.DataFrame({
    "fish": ["/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"]
})


signature = ModelSignature(
    inputs=Schema([ColSpec("string", "fish")]),
    outputs=Schema([
        ColSpec("integer", "fish_id"),
        ColSpec("string", "video"),
        ColSpec("integer", "frame"),
        ColSpec("float", "x1"),
        ColSpec("float", "y1"),
        ColSpec("float", "x2"),
        ColSpec("float", "y2"),
        ColSpec("float", "confidence")
    ])
)

In [0]:
with mlflow.start_run() as run:
    run_id = run.info.run_id

    mlflow.pyfunc.log_model(
        name="model",  # 
        python_model=FishVideoDetector(),
        input_example=input_example,
        artifacts={"checkpoint": model_path},
        signature=signature
    )

print("Logged model under run:", run_id)


### Register model

In [0]:
mlflow.set_registry_uri("databricks-uc")

In [0]:
result = mlflow.register_model(
    model_uri=f"runs:/{run_id}/model",
    name="prd_dash_lab.dash_data_science_unrestricted.FishVideoDetector"
)

print("Registered model version:", result.version)

### Test registered model ( I didnt test for UC model since endpoints in UC models serving does not work yet. Skip to workspace registry)

In [0]:
model = mlflow.pyfunc.load_model("models:/prd_dash_lab.dash_data_science_unrestricted.salmon_model_test/9")

In [0]:
import pandas as pd
import os

# Directory where images are stored
image_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/"

# List all files (filter for jpg/png)
image_files = [f for f in os.listdir(image_dir) if f.endswith((".mp4"))]

# Build full paths
full_paths = [os.path.join(image_dir, f) for f in image_files]

# Create a DataFrame suitable for your model
input_df = pd.DataFrame({
    "fish": full_paths
})

print(input_df.head())


In [0]:
results = model.predict(input_df)


In [0]:
display(results)

### Register model to workspace

In [0]:
import mlflow

mlflow.set_registry_uri("databricks")

model_uri = f"runs:/{run_id}/model"

print("About to register model from:")
print("Run ID:", run_id)
print("Model URI:", model_uri)

result = mlflow.register_model(
    model_uri=model_uri,
    name="FishVideoDetector"
)

print("\nRegistered model version:", result.version)
print("Registered from run ID:", result.run_id)

# Extra safety check
if result.run_id == run_id:
    print("Registration confirmed: correct run ID used.")
else:
    print(" WARNING: Registered run ID does NOT match expected run ID!")



In [0]:
import mlflow.pyfunc
import pandas as pd

mlflow.set_registry_uri("databricks")

model_uri = "models:/FishVideoDetector/2"
model = mlflow.pyfunc.load_model(model_uri)

input_df = pd.DataFrame({
    "fish": [
        "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
    ]
})

preds = model.predict(input_df)
print(preds)


In [0]:
print(preds)

In [0]:
%pip install -r /local_disk0/repl_tmp_data/ReplId-19c33-9d7a9-0/tmp7dl33gm3/requirements.txt